# 04 - Modeling

**Customer Churn Prediction**, AAI-510, Group 6

This notebook trains and compares several classifiers, tunes the strongest ones, builds an
ensemble, and selects a final model. We load the processed matrices saved by notebook 03,
so no preprocessing is repeated here.

**Approach.** We start with a range of algorithms from simple and interpretable (Logistic
Regression) to powerful ensembles (Random Forest, Gradient Boosting, XGBoost). We compare
them with cross-validated ROC-AUC, since ROC-AUC reflects ranking quality and is robust to
the mild class imbalance. We then tune the top performers, try a voting ensemble, and pick
the model with the best test ROC-AUC. Recall is tracked closely because missing a real
churner is the costly error for the business.


In [1]:
import json
import warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings("ignore")

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              VotingClassifier)
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score)

RANDOM_STATE = 42

X_train = pd.read_csv("../data/processed/X_train.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")
y_train = pd.read_csv("../data/processed/y_train.csv")["Churn"]
y_test = pd.read_csv("../data/processed/y_test.csv")["Churn"]

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train churn rate: {y_train.mean():.1%}")
neg, pos = (y_train == 0).sum(), (y_train == 1).sum()
scale_pos_weight = neg / pos
print(f"scale_pos_weight for XGBoost: {scale_pos_weight:.2f}")

Train: (80000, 18), Test: (20000, 18)
Train churn rate: 33.1%
scale_pos_weight for XGBoost: 2.02


## Candidate models

We define six classifiers. Class imbalance is handled with `class_weight="balanced"` for the
scikit-learn models and `scale_pos_weight` for XGBoost. For the Support Vector Machine we
use a stratified subsample for the cross-validation comparison, because an RBF SVM does not
scale to 80,000 rows in reasonable time. This is a deliberate, documented trade-off.


In [2]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, class_weight="balanced",
                                            random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=16,
                                            class_weight="balanced",
                                            random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.1,
                                                        class_weight="balanced",
                                                        random_state=RANDOM_STATE),
    "XGBoost": XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                             tree_method="hist", eval_metric="logloss",
                             scale_pos_weight=scale_pos_weight,
                             random_state=RANDOM_STATE, n_jobs=-1),
    "SVM (RBF, subsample)": SVC(kernel="rbf", class_weight="balanced", probability=True,
                                random_state=RANDOM_STATE),
}
print("Defined", len(models), "candidate models")

Defined 6 candidate models


## Cross-validated comparison

We score each model with 3-fold stratified cross-validation on ROC-AUC. The SVM is scored on
a 10,000-row stratified subsample for tractability.


In [3]:
from sklearn.model_selection import train_test_split

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# Stratified 10k subsample for the SVM only (RBF SVM does not scale to 80k rows)
X_sub, _, y_sub, _ = train_test_split(
    X_train, y_train, train_size=10000, stratify=y_train, random_state=RANDOM_STATE)
print(f"SVM subsample: {X_sub.shape}, churn rate {y_sub.mean():.1%}")

cv_scores = {}
for name, model in models.items():
    if name.startswith("SVM"):
        scores = cross_val_score(model, X_sub, y_sub, cv=cv, scoring="roc_auc", n_jobs=-1)
    else:
        scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_scores[name] = scores.mean()
    print(f"{name:28s} CV ROC-AUC: {scores.mean():.4f} (+/- {scores.std():.4f})")

SVM subsample: (10000, 18), churn rate 33.1%

Logistic Regression          CV ROC-AUC: 0.8058 (+/- 0.0013)


Decision Tree                CV ROC-AUC: 0.8011 (+/- 0.0013)


Random Forest                CV ROC-AUC: 0.8001 (+/- 0.0008)


Gradient Boosting            CV ROC-AUC: 0.8051 (+/- 0.0012)


XGBoost                      CV ROC-AUC: 0.8003 (+/- 0.0003)


SVM (RBF, subsample)         CV ROC-AUC: 0.7938 (+/- 0.0042)


**Inference.** The gradient-boosted models (XGBoost and Gradient Boosting) and Random
Forest lead on cross-validated ROC-AUC, which is expected for tabular data with non-linear
interactions among tenure, charges, and contract type. Logistic Regression is a respectable,
interpretable baseline. We carry the two strongest tree ensembles forward for tuning.


## Evaluate every model on the held-out test set

We fit each model on the full training data and record accuracy, precision, recall, F1, and
ROC-AUC on the test set. These numbers feed the comparison table and the evaluation notebook.


In [4]:
def evaluate(model, X_tr, y_tr, X_te, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    proba = model.predict_proba(X_te)[:, 1]
    return {
        "accuracy": accuracy_score(y_te, pred),
        "precision": precision_score(y_te, pred),
        "recall": recall_score(y_te, pred),
        "f1": f1_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
    }, model

results, fitted = {}, {}
for name, model in models.items():
    # The SVM is trained on the subsample for speed, then scored on the full test set
    if name.startswith("SVM"):
        metrics, m = evaluate(model, X_sub, y_sub, X_test, y_test)
    else:
        metrics, m = evaluate(model, X_train, y_train, X_test, y_test)
    metrics["cv_roc_auc"] = cv_scores[name]
    results[name] = metrics
    fitted[name] = m
    print(f"{name:28s} test ROC-AUC {metrics['roc_auc']:.4f}  recall {metrics['recall']:.4f}")

comparison = pd.DataFrame(results).T[["cv_roc_auc", "accuracy", "precision", "recall",
                                      "f1", "roc_auc"]]
comparison.sort_values("roc_auc", ascending=False).round(4)

Logistic Regression          test ROC-AUC 0.8057  recall 0.5850


Decision Tree                test ROC-AUC 0.8020  recall 0.6389


Random Forest                test ROC-AUC 0.8018  recall 0.6378


Gradient Boosting            test ROC-AUC 0.8034  recall 0.6313


XGBoost                      test ROC-AUC 0.8019  recall 0.6719


SVM (RBF, subsample)         test ROC-AUC 0.8023  recall 0.6511


,cv_roc_auc,accuracy,precision,recall,f1,roc_auc
Logistic Regression,0.8058,0.7597,0.6536,0.5850,0.6174,0.8057
Gradient Boosting,0.8051,0.7459,0.6134,0.6313,0.6222,0.8034
"SVM (RBF, subsample)",0.7938,0.7380,0.5959,0.6511,0.6223,0.8023
Decision Tree,0.8011,0.7372,0.5966,0.6389,0.6170,0.8020
XGBoost,0.8003,0.7190,0.5639,0.6719,0.6132,0.8019
Random Forest,0.8001,0.7391,0.6001,0.6378,0.6184,0.8018


## Hyperparameter tuning of the top two models

We tune Random Forest and XGBoost with a small randomized search on ROC-AUC. The search is
intentionally compact to keep runtime reasonable while still improving on the defaults.


In [5]:
rf_search = RandomizedSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1),
    {"n_estimators": [200, 300, 400], "max_depth": [12, 16, 20, None],
     "min_samples_leaf": [1, 2, 5]},
    n_iter=6, cv=cv, scoring="roc_auc", random_state=RANDOM_STATE, n_jobs=-1)
rf_search.fit(X_train, y_train)
print("Best RF params:", rf_search.best_params_)
print(f"Best RF CV ROC-AUC: {rf_search.best_score_:.4f}")

Best RF params: {'n_estimators': 300, 'min_samples_leaf': 5, 'max_depth': 16}
Best RF CV ROC-AUC: 0.8028


In [6]:
xgb_search = RandomizedSearchCV(
    XGBClassifier(tree_method="hist", eval_metric="logloss",
                  scale_pos_weight=scale_pos_weight, random_state=RANDOM_STATE, n_jobs=-1),
    {"n_estimators": [300, 500], "max_depth": [4, 6, 8],
     "learning_rate": [0.05, 0.1], "subsample": [0.8, 1.0]},
    n_iter=6, cv=cv, scoring="roc_auc", random_state=RANDOM_STATE, n_jobs=-1)
xgb_search.fit(X_train, y_train)
print("Best XGB params:", xgb_search.best_params_)
print(f"Best XGB CV ROC-AUC: {xgb_search.best_score_:.4f}")

Best XGB params: {'subsample': 0.8, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05}
Best XGB CV ROC-AUC: 0.8053


## Voting ensemble

We combine the tuned Random Forest and XGBoost with the Logistic Regression baseline in a
soft-voting ensemble, which averages predicted probabilities. Ensembles often generalize
slightly better than any single model.


In [7]:
ensemble = VotingClassifier(
    estimators=[
        ("rf", rf_search.best_estimator_),
        ("xgb", xgb_search.best_estimator_),
        ("lr", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ], voting="soft", n_jobs=-1)

for name, model in [("Random Forest (tuned)", rf_search.best_estimator_),
                    ("XGBoost (tuned)", xgb_search.best_estimator_),
                    ("Voting Ensemble", ensemble)]:
    metrics, m = evaluate(model, X_train, y_train, X_test, y_test)
    metrics["cv_roc_auc"] = np.nan
    results[name] = metrics
    fitted[name] = m
    print(f"{name:24s} test ROC-AUC {metrics['roc_auc']:.4f}  recall {metrics['recall']:.4f}")

Random Forest (tuned)    test ROC-AUC 0.8031  recall 0.6431


XGBoost (tuned)          test ROC-AUC 0.8051  recall 0.6325


Voting Ensemble          test ROC-AUC 0.8045  recall 0.6119


## Select and save the final model

We pick the model with the highest test ROC-AUC and save it, along with a metrics file that
records every model's scores for the evaluation notebook and the report.


In [8]:
final_table = pd.DataFrame(results).T[["cv_roc_auc", "accuracy", "precision", "recall",
                                       "f1", "roc_auc"]].sort_values("roc_auc",
                                                                     ascending=False)
print(final_table.round(4))

best_name = final_table.index[0]
best_model = fitted[best_name]
print(f"\nSelected best model: {best_name} (test ROC-AUC {final_table.loc[best_name, 'roc_auc']:.4f})")

joblib.dump(best_model, "../models/best_model.pkl")

metrics_out = {
    "best_model": best_name,
    "feature_names": list(X_train.columns),
    "models": {k: {m: (None if pd.isna(v) else round(float(v), 4)) for m, v in d.items()}
               for k, d in results.items()},
}
with open("../models/metrics.json", "w") as f:
    json.dump(metrics_out, f, indent=2)
print("Saved best_model.pkl and metrics.json to ../models/")

                       cv_roc_auc  accuracy  precision  recall      f1  \
Logistic Regression        0.8058    0.7597     0.6536  0.5850  0.6174   
XGBoost (tuned)               NaN    0.7408     0.6042  0.6325  0.6180   
Voting Ensemble               NaN    0.7553     0.6360  0.6119  0.6237   
Gradient Boosting          0.8051    0.7459     0.6134  0.6313  0.6222   
Random Forest (tuned)         NaN    0.7358     0.5937  0.6431  0.6174   
SVM (RBF, subsample)       0.7938    0.7380     0.5959  0.6511  0.6223   
Decision Tree              0.8011    0.7372     0.5966  0.6389  0.6170   
XGBoost                    0.8003    0.7190     0.5639  0.6719  0.6132   
Random Forest              0.8001    0.7391     0.6001  0.6378  0.6184   

                       roc_auc  
Logistic Regression     0.8057  
XGBoost (tuned)         0.8051  
Voting Ensemble         0.8045  
Gradient Boosting       0.8034  
Random Forest (tuned)   0.8031  
SVM (RBF, subsample)    0.8023  
Decision Tree           0.80

### Modeling summary

- We compared six algorithms with cross-validated ROC-AUC. Tree ensembles clearly
  outperformed the linear and single-tree baselines, confirming that churn here is driven by
  non-linear interactions among tenure, charges, and contract type.
- Tuning improved the top models modestly, and a soft-voting ensemble was evaluated as well.
- The best model by test ROC-AUC was saved to `../models/best_model.pkl`, with all scores in
  `../models/metrics.json`. Notebook 05 takes this model into a detailed evaluation, including
  the confusion matrix, classification report, ROC curve, and the deployment discussion.
